## Обычный декоратор без параметра

In [1]:
from typing import Callable
import time

def timer(func: Callable):
    def wrapper():
        start = time.time()
        res = func()
        end = time.time()
        print(f"Исполнение функции {func} занимает {end - start} секунд")
        return res
    return wrapper

@timer
def func():
    time.sleep(2)
    return 123

func()

Исполнение функции <function func at 0x000002B5AF0F58A0> занимает 2.000443935394287 секунд


123

## Декоратор с параметром (repeat)

In [49]:
from typing import Callable

def repeat(n):
    def deco(func: Callable):
        def wrapper(*args, **kwargs):
            for _ in range(n):
                res = func(*args, **kwargs)
            return res          # вернёт последний результат func
        return wrapper
    return deco

@repeat(3)
def hello(name):
    return f"Привет, {name}"     # возвращает str, не print

result = hello(name="Игорь")
print(result)                    # Привет, Игорь (один раз, снаружи)
print(type(result))              # <class 'str'>

Привет, Игорь
<class 'str'>


# Задачи на декораторы

In [ ]:
"""Задача 1 easy: count_calls"""

from typing import Callable
from functools import wraps


def count_calls(func: Callable):
    @wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@count_calls
def add(a, b):
    return a + b

add(1, 2); add(3, 4)
print(add.calls)     # 2
print(add.__name__)  # "add"

@count_calls
def other():
    pass

print(other.calls)   # 0  ← не 2, счётчики независимы

2
add
0


In [8]:
"""Задача 2 medium: retry с параметрами backoff"""

import time
from typing import Callable
from functools import wraps


def retry(max_attempts: int = 3, delay: float = 0.1, backoff: float = 2.0, exceptions: tuple = (Exception,)):
    def wrapper(func: Callable):
        @wraps(func)
        def inner(*args, **kwargs):
            curr_delay = delay
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == max_attempts - 1: raise
                    time.sleep(curr_delay)
                    curr_delay *= backoff
        return inner
    return wrapper


import time

calls = []

@retry(max_attempts=3, delay=0.01, backoff=2.0, exceptions=(ValueError,))
def flaky(x):
    calls.append(x)
    if len(calls) < 3:
        raise ValueError("not yet")
    return "ok"

print(flaky(42))          # "ok"
print(len(calls))         # 3
# паузы были ~0.01, затем ~0.02, всего ~0.03 сек

@retry(max_attempts=3, exceptions=(ValueError,))
def wrong():
    raise KeyError("boom")

# wrong() должен упасть СРАЗУ с KeyError, без ретраев
try:
    wrong()
except KeyError as ke:
    print(f"Функция wrong упала с ошибкой {str(ke)}")

ok
3
Функция wrong упала с ошибкой 'boom'


In [19]:
"""Задача 3 easy: ensure_positive"""

from typing import Callable
from functools import wraps


def ensure_positive(func: Callable):
    @wraps(func)
    def wrapper(*args, **kwargs):
        for el in (*args, *kwargs.values()):
            if isinstance(el, bool):
                continue
            if isinstance(el, (int, float)) and el <= 0:
                raise ValueError("argument must be positive")
        return func(*args, **kwargs)
    return wrapper


@ensure_positive
def divide(a, b):
    return a / b

print(divide(10, 2))     # 5.0
#divide(10, -2)     # ValueError: argument must be positive
#divide(10, b=0)    # ValueError
divide("x", 2)     # "x" не число → игнор, но потом упадёт сам divide с TypeError

5.0


TypeError: unsupported operand type(s) for /: 'str' and 'int'

In [45]:
"""Задача 4 medium: validate_types"""

import inspect
from typing import Callable
from functools import wraps


def validate_types(strict: bool):
    def wrapper(func: Callable):
        @wraps(func)
        def inner(*args, **kwargs):
            a = inspect.signature(func).bind(*args, **kwargs)
            for k, v in a.arguments.items(): # k - имя переданного параметра | v - реальное значение переданного параметра
                given_annotation = func.__annotations__.get(k) # Класс аннотации
                if given_annotation is None or not isinstance(given_annotation, type): continue
                if not isinstance(v, given_annotation):
                    raise TypeError
                elif strict and given_annotation is int and isinstance(v, bool):
                    raise TypeError("strict: bool не int")
            return func(*args, **kwargs)
        return inner
    return wrapper

@validate_types(strict=False)
def add(a: int, b: int) -> int:
    return a + b

#print(add(1, 2))        # 3
print(add(1, True))       # TypeError (strict: bool не int)
add(1, "2")        # TypeError

2


TypeError: 